In [1]:
!pip install -U --quiet \
  langchain==0.3.27 \
  langchain-core==0.3.79 \
  langchain-google-genai==2.0.8 \
  langgraph==0.2.60 \
  chromadb==0.6.3 \
  langchain-groq==0.2.4 \
  rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.8/449.8 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip show langchain-core langchain langchain-google-genai langgraph chromadb | grep -E "^(Name|Version)"

Name: langchain-core
Version: 0.3.79
Name: langchain
Version: 0.3.27
Name: langchain-google-genai
Version: 2.0.8
Name: langgraph
Version: 0.2.60
Name: chromadb
Version: 0.6.3


In [3]:
!pip install groq

In [4]:
import re
import json
from typing import List, Dict
from collections import defaultdict

import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi

import google.generativeai as genai
import os
from groq import Groq


/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  # the post import hooks.


In [62]:
keys

In [6]:
import shutil
import os

# Source folder (read-only, uploaded)
SRC_DB_FOLDER = "/kaggle/input/datasets/hamnainam/chromadbv2"

# Destination folder (writable)
DST_DB_FOLDER = "/kaggle/working/chromadbv2"

# Copy entire folder
if not os.path.exists(DST_DB_FOLDER):
    shutil.copytree(SRC_DB_FOLDER, DST_DB_FOLDER)
    print(f"✅ DB copied to writable folder: {DST_DB_FOLDER}")
else:
    print(f"⚠️ Destination already exists: {DST_DB_FOLDER}")

# Verify contents
print("Files in writable folder:", os.listdir(DST_DB_FOLDER))


✅ DB copied to writable folder: /kaggle/working/chromadbv2
Files in writable folder: ['9290ce11-db52-4e59-80e6-e06778e58b7e', 'chroma.sqlite3']


# Retrieval Pipeline

In [56]:
import re
import json
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi
from typing import List, Dict, Optional
from collections import defaultdict

# ============================================
# GLOBALS
# ============================================

_client = None
_collection = None
_CHUNKS = None

# ============================================
# INIT
# ============================================

def _init_db():
    global _client, _collection, _CHUNKS

    if _collection is not None:
        return

    _client = chromadb.PersistentClient(path=CHROMA_PATH)
    embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name=EMBEDDING_MODEL_NAME
    )
    _collection = _client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=embed_fn
    )

    res = _collection.get()
    _CHUNKS = [
        {
            "chunk_id": res["ids"][i],
            "text": res["documents"][i],
            "metadata": res["metadatas"][i]
        }
        for i in range(len(res["ids"]))
    ]
    print(f"✅ DB loaded: {len(_CHUNKS)} chunks")

# ============================================
# RETRIEVAL HELPERS
# ============================================

def _tokenize(text: str) -> List[str]:
    return re.findall(r"\w+", text.lower())

def _dedup_by_case(results: List[Dict]) -> List[Dict]:
    seen = {}
    for r in results:
        cid = r["metadata"].get("case_id")
        if cid and cid not in seen:
            seen[cid] = r
    return list(seen.values())

    
def _normalize_filter_values(metadata_filter: dict) -> dict:
    
    if not metadata_filter:
        return None
    normalized = {}
    for k, v in metadata_filter.items():
        if v is None or str(v).lower() == "none":  # ← skip null values
            continue
        if k == "year" and not str(v).endswith(".0"):
            v = f"{v}.0"
        if k == "judges":
            v = re.sub(
                r"^(MR\.\s+JUSTICE\s+|MRS\.\s+JUSTICE\s+|Mr\.\s+Justice\s+|Mrs\.\s+Justice\s+|JUSTICE\s+|Justice\s+|MR\.\s+|MRS\.\s+)",
                "", v, flags=re.IGNORECASE
            ).strip()
        normalized[k] = v
    return normalized if normalized else None

def _format_chroma_filter(metadata_filter: dict) -> dict:
    if not metadata_filter:
        return None

    # These fields need substring matching — only BM25 handles them
    bm25_only_fields = {"judges", "petitioner", "respondent"}

    items = [(k, v) for k, v in metadata_filter.items() if k not in bm25_only_fields]

    print(f"  chroma items: {items}") 

    if not items:
        return None
    if len(items) == 1:
        k, v = items[0]
        return {k: {"$eq": v}}
    else:
        return {"$and": [{k: {"$eq": v}} for k, v in items]}

def _dense_search(query: str, metadata_filter: dict = None, k: int = 30) -> List[Dict]:
    _init_db()
    try:
        if metadata_filter:
            res = _collection.query(
                query_texts=[query],
                where=metadata_filter,
                n_results=k
            )
        else:
            res = _collection.query(
                query_texts=[query],
                n_results=k
            )
    except Exception as e:
        print(f"  ⚠️ Dense search error: {e}")
        return []

    results = [
        {
            "text": res["documents"][0][i],
            "metadata": res["metadatas"][0][i],
            "score": 1 - res["distances"][0][i],
            "method": "DENSE"
        }
        for i in range(len(res["ids"][0]))
    ]
    return _dedup_by_case(results)

def _bm25_search_filtered(query: str, bm25_filter: dict = None, k: int = 30) -> List[Dict]:
    
    _init_db()
    if not isinstance(bm25_filter, dict):
        bm25_filter = None
    if bm25_filter:
        filtered_chunks = [
            ch for ch in _CHUNKS
            if all(
                str(v).lower() in str(ch["metadata"].get(fk, "")).lower()
                for fk, v in bm25_filter.items()
            )
        ]
    else:
        filtered_chunks = _CHUNKS

    if not filtered_chunks:
        print("  ⚠️ BM25: No chunks matched metadata filter")
        return []

    corpus = [_tokenize(ch["text"]) for ch in filtered_chunks]
    bm25 = BM25Okapi(corpus)
    scores = bm25.get_scores(_tokenize(query))
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:k]
    results = [
        {**filtered_chunks[idx], "score": score, "method": "BM25"}
        for idx, score in ranked
    ]
    return _dedup_by_case(results)

def _aggregate_rrf(dense_results: List[Dict], bm25_results: List[Dict]) -> List[Dict]:
    scores = defaultdict(lambda: {
        "case_id": None,
        "rrf_score": 0.0,
        "matched_on": set(),
        "preview": ""
    })

    k = 60

    for rank, ch in enumerate(dense_results):
        cid = ch["metadata"].get("case_id")
        if not cid:
            continue
        scores[cid]["case_id"] = cid
        scores[cid]["rrf_score"] += 1 / (k + rank + 1)
        scores[cid]["matched_on"].add("DENSE")
        if not scores[cid]["preview"]:
            scores[cid]["preview"] = ch["text"][:150]

    for rank, ch in enumerate(bm25_results):
        cid = ch["metadata"].get("case_id")
        if not cid:
            continue
        scores[cid]["case_id"] = cid
        scores[cid]["rrf_score"] += 1 / (k + rank + 1)
        scores[cid]["matched_on"].add("BM25")
        if not scores[cid]["preview"]:
            scores[cid]["preview"] = ch["text"][:150]

    return sorted(
        [
            {
                "case_id": v["case_id"],
                "score": round(v["rrf_score"], 4),
                "matched_on": list(v["matched_on"]),
                "preview": v["preview"]
            }
            for v in scores.values()
        ],
        key=lambda x: x["score"],
        reverse=True
    )

def _filter_relative(cases: List[Dict], drop_threshold: float = 0.5, top_k: int = 3) -> List[Dict]:
    if not cases:
        return []
    top_score = cases[0]["score"]
    return [c for c in cases if c["score"] >= top_score * drop_threshold][:top_k]

def _build_chunks(res) -> List[Dict]:
    chunks = []
    if not res or not res["ids"] or not res["ids"][0]:
        return chunks
    for i, cid in enumerate(res["ids"][0]):
        chunks.append({
            "chunk_id": cid,
            "text": res["documents"][0][i],
            "metadata": res["metadatas"][0][i],
            "score": 1 - res["distances"][0][i]
        })
    return chunks

def _resolve_case_id(case_id: str) -> str:
    """Handle SC-PK_ prefix variants."""
    _init_db()
    all_case_ids = set(ch["metadata"]["case_id"] for ch in _CHUNKS if ch["metadata"].get("case_id"))
    
    if case_id in all_case_ids:
        return case_id
    
    prefixed = f"SC-PK_{case_id}"
    if prefixed in all_case_ids:
        return prefixed
    
    if case_id.startswith("SC-PK_"):
        stripped = case_id[6:]
        if stripped in all_case_ids:
            return stripped
    
    return case_id

def _expand_case(case_id: str, k: int = 15) -> List[Dict]:
    _init_db()
    try:
        resolved_id = _resolve_case_id(case_id)
        if resolved_id != case_id:
            print(f"  🔧 Resolved case_id: {case_id} → {resolved_id}")
        res = _collection.query(
            query_texts=[""],
            where={"case_id": resolved_id},
            n_results=k
        )
        return _build_chunks(res)
    except Exception as e:
        print(f"  ⚠️ expand_case error: {e}")
        return []

# ============================================
# FORMATTERS
# ============================================

def _format_context_for_llm(chunks: List[Dict]) -> str:
    context = ""
    for i, ch in enumerate(chunks):
        m = ch["metadata"]
        context += (
            f"### DATA_BLOCK_{i}\n"
            f"CASE_ID: {m.get('case_id')}\n"
            f"TEXT:\n{ch['text']}\n"
            f"### END_BLOCK_{i}\n\n"
        )
    return context

def _format_context_for_verifier(chunks: List[Dict]) -> str:
    context = ""
    for i, ch in enumerate(chunks):
        context += f"""
[CONTEXT_BLOCK_{i}]
CASE_ID: {ch['metadata'].get('case_id')}
TEXT:
{ch['text']}
[END_CONTEXT_BLOCK_{i}]
"""
    return context

# ============================================
# LLM CALLS
# ============================================
def _call_groq_json(prompt: str, system: str = None) -> dict:
    """Call Groq and parse JSON response reliably."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0
    )
    return json.loads(response.choices[0].message.content)

def _call_groq_text(prompt: str, system: str = None) -> str:
    """Call Groq and return plain text response."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content.strip()

def _parse_legal_query(user_query: str) -> dict:
    system = """You are a Pakistan Legal Research Assistant. Convert user queries into a JSON object for a hybrid search system.

CLASSIFICATION RULES:
1. PURE METADATA: If the user searches for a specific person (Justice/Judge), use metadata_filter and keep semantic_query brief.
2. HYBRID: If the user specifies a topic AND a year/judge/party, use both fields.
3. PURE SEMANTIC: For legal concepts, leave metadata_filter as null and expand semantic_query with legal synonyms.

DATA FORMATS:
- Years: Must be strings ending in .0 (e.g. 2023.0)
- Judges: Use formal titles (e.g. MR. JUSTICE YAHYA AFRIDI)
- Case Types: Crl.A (Criminal Appeal), C.A (Civil Appeal), C.P (Constitutional Petition)

METADATA FILTER FIELDS:
- year: e.g. "2023.0"
- judges: e.g. "YAHYA AFRIDI"
- case_type: e.g. "C.A", "Crl.A"
- petitioner: ONLY if query explicitly names the party filing the case (e.g. "Dawood Investment Bank", "Sardar Khan")
- respondent: ONLY if query explicitly names the opposing party

STRICT RULES:
- DO NOT include case_id in metadata_filter
- DO NOT invent or assume party names not explicitly stated in the query
- DO NOT add petitioner/respondent unless the query contains a specific person or organization name as a party
- If a name appears in the query but you are unsure if they are petitioner or respondent, put them in petitioner only

EXAMPLES:
Query: "Section 302 PPC murder cases"
Response: {"semantic_query": "Section 302 PPC murder qatl-i-amd homicide conviction", "metadata_filter": null}

Query: "Recent tax cases from 2023"
Response: {"semantic_query": "tax taxation revenue FBR customs", "metadata_filter": {"year": "2023.0"}}

Query: "Cases by Justice Munib Akhtar"
Response: {"semantic_query": "Justice Munib Akhtar", "metadata_filter": {"judges": "MR. JUSTICE MUNIB AKHTAR"}}

Query: "Cases involving Dawood Investment Bank"
Response: {"semantic_query": "Dawood Investment Bank financial institution case", "metadata_filter": {"petitioner": "Dawood Investment Bank"}}

Query: "Cases where Sardar Khan was acquitted"
Response: {"semantic_query": "acquittal conviction overturned jail petition", "metadata_filter": {"petitioner": "Sardar Khan"}}

Query: "What did the court rule about property rights in 2022?"
Response: {"semantic_query": "property rights ownership title inheritance dispute 2022", "metadata_filter": {"year": "2022.0"}}

Return ONLY a JSON object with keys: semantic_query, metadata_filter"""

    result = _call_groq_json(
        f"Query: {user_query}",
        system=system
    )
    print(f"  🔍 Parsed intent: {result}")

    raw_filter = result.get("metadata_filter")
    normalized_filter = _normalize_filter_values(raw_filter)
    chroma_filter = _format_chroma_filter(normalized_filter)

    print(f"  normalized_filter: {normalized_filter}")
    print(f"  chroma_filter: {chroma_filter}")
    print(f"  bm25_filter: {normalized_filter}")

    return {
        "semantic_query": result.get("semantic_query", user_query),
        "metadata_filter": chroma_filter,
        "bm25_filter": normalized_filter
    }

def _detect_case_reference(query: str) -> Optional[str]:
    prompt = f"""You are analyzing a legal query to detect case number references.

Query: {query}

Does this query mention a SPECIFIC case number?

Common formats to recognize:
- "Civil Appeal No. 875 of 2017" → C.A.875_2017
- "Criminal Appeal No. 456 of 2018" → Crl.A.456_2018
- "C.A.123-2020" → C.A.123_2020
- "Crl.P.L.A.645-L_2025" → Crl.P.L.A.645-L_2025 (preserve -L, -K, -P suffixes)
- "Civil Petition No. 123 of 2019" → C.P.L.A.123_2019

IMPORTANT: Preserve any suffixes like -L (Lahore), -K (Karachi), -P (Peshawar) exactly as written.
Normalized format: [TYPE].[NUMBER]_[YEAR] or [TYPE].[NUMBER]-[SUFFIX]_[YEAR]

Respond with ONLY this JSON:
{{"has_case_id": true, "case_id": "Crl.P.L.A.645-L_2025"}}
OR
{{"has_case_id": false, "case_id": null}}"""

    try:
        response = gemini_model.generate_content(
            prompt,
            generation_config={"temperature": 0}
        )
        cleaned = re.sub(r"```json|```", "", response.text).strip()
        result = json.loads(cleaned)
        return result.get("case_id") if result.get("has_case_id") else None
    except Exception as e:
        print(f"  ⚠️ detect_case_reference failed: {e}, returning None")
        return None

def _rewrite_query(query: str) -> str:
    system = """You are a legal research expert. Rewrite queries to improve retrieval from a Pakistani Supreme Court case database.

RULES:
1. PRESERVE EXACTLY: Any case numbers (e.g. C.A.875_2017)
2. PRESERVE: Question format (What/Why/How/Did/When)
3. PRESERVE: Specific names, judges, locations, parties
4. ADD: English legal synonyms and related terms only
5. ADD: Expanded abbreviations (PPC → Pakistan Penal Code, ECP → Election Commission of Pakistan)
6. DO NOT: Add Urdu translations
7. DO NOT: Make it a keyword dump
8. Keep rewritten query under 30 words if possible

Return ONLY the rewritten query as plain text, no explanation."""

    return _call_groq_text(
        f"Original query: {query}",
        system=system
    )

def _generate_answer(query: str, chunks: List[Dict]) -> str:
    context = _format_context_for_llm(chunks)

    # Truncate context if too large
    max_context_chars = 8000
    if len(context) > max_context_chars:
        context = context[:max_context_chars]
        print(f"  ⚠️ Context truncated to {max_context_chars} chars")

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are an Expert Legal Research Assistant answering questions about Pakistani Supreme Court case law.

STRICT PROCEDURE (MANDATORY):
1. Identify which CASE_ID is most relevant to the question.
2. If no block from any single case answers the query output Context insufficient.
3. Answer using ONLY blocks that belong to that ONE case_id.
   You may combine information from multiple blocks of the same case if needed.

IMPORTANT:
- Do NOT use blocks from different cases.
- Do NOT make up information not present in the blocks.
- Legal implication is sufficient, verbatim wording not required.

MANDATORY RESPONSE FORMAT (NO DEVIATION):

Selected Case: [CASE_ID]

Answer: [Clear legal answer in your own words]

Case Id: [CASE_ID]

Relevant chunks: [Exact sentence(s) from the selected blocks supporting the answer]

FAILURE CONDITION:
- If you cannot find a relevant case respond exactly with:
Answer: Context insufficient
Case Id: NONE
Relevant chunks: NONE"""
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()
def _generate_answer(query: str, chunks: List[Dict]) -> str:
    context = _format_context_for_llm(chunks)

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are an Expert Legal Research Assistant answering questions about Pakistani Supreme Court case law.

STRICT PROCEDURE (MANDATORY):
1. Identify which CASE_ID is most relevant to the question.
2. If no block from any single case answers the query → output "Context insufficient".
3. Answer using ONLY blocks that belong to that ONE case_id.
   You may combine information from multiple blocks of the same case if needed.

IMPORTANT:
- Do NOT use blocks from different cases.
- Do NOT make up information not present in the blocks.
- Legal implication is sufficient — verbatim wording not required.

MANDATORY RESPONSE FORMAT (NO DEVIATION):

Selected Case: [CASE_ID]

Answer: [Clear legal answer in your own words]

Case Id: [CASE_ID]

Relevant chunks: [Exact sentence(s) from the selected blocks supporting the answer]

FAILURE CONDITION:
- If you cannot find a relevant case, respond exactly with:
Answer: Context insufficient
Case Id: NONE
Relevant chunks: NONE"""
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()

def _verify_with_gemini(query: str, answer: str, chunks: List[Dict]) -> bool:
    try:
        ans_part = answer.split("Answer:", 1)[1].split("Case Id:", 1)[0].strip()
    except:
        print("❌ Could not parse answer format")
        return False

    full_context_text = _format_context_for_verifier(chunks)

    verification_prompt = f"""You are verifying if a legal answer is grounded in the provided case law context.

QUESTION:
{query}

PROPOSED ANSWER:
{ans_part}

CONTEXT BLOCKS (all from the SAME case):
{full_context_text}

VERIFICATION STEPS (follow in strict order):

STEP 1 — CASE RELEVANCE CHECK (mandatory gate):
- Extract ALL specific entities from the QUESTION: people, judges, organizations, case numbers
- Check if these entities appear anywhere in the context blocks
- If ANY key entity from the question is missing from ALL blocks → verdict is NO, STOP

STEP 2 — ANSWER GROUNDING CHECK (only if Step 1 passes):
- Check if the proposed answer is supported by the context blocks
- Paraphrasing is acceptable — look for substance, not exact wording
- The answer is NOT supported if:
  * The blocks contradict the answer
  * The answer makes specific claims not found anywhere in the blocks

CRITICAL RULES:
- These are blocks from ONE case only — if the question is about a different case, verdict is NO
- A legally correct answer from the wrong case is WRONG — verdict is NO
- Both steps must pass for verdict to be YES

Respond with ONLY this JSON (no explanation):
{{"verdict": "YES"}} or {{"verdict": "NO"}}"""

    try:
        response = gemini_model.generate_content(
            verification_prompt,
            generation_config={"temperature": 0}
        )
        cleaned = re.sub(r"```json|```", "", response.text).strip()
        result = json.loads(cleaned)
        verdict = result.get("verdict", "NO")
        print(f"  ✅ VERDICT: {verdict}")
        return verdict == "YES"
    except Exception as e:
        print(f"  ❌ Verification error: {e}")
        return False

print("✅ Retrieval & QA functions ready")

✅ Retrieval & QA functions ready


In [43]:
# ============================================
# PLANNER PROMPTS
# ============================================

CASE_SEARCH_PLANNER_PROMPT = """You are a Planner for a Pakistani Supreme Court case search system.

You will receive a user query and must output a JSON plan — an ordered list of tool calls to find relevant cases.

AVAILABLE TOOLS:
- parse_legal_query: Parses query into semantic_query, metadata_filter (for ChromaDB), bm25_filter (for BM25). Always call this first.
- dense_search: Semantic embedding search. Args: query (str), metadata_filter (dict or null), k (int, default 30)
- bm25_search: Keyword search. Args: query (str), bm25_filter (dict or null), k (int, default 30). Handles judges filtering since ChromaDB cannot.
- aggregate_rrf: Combines dense and BM25 results using Reciprocal Rank Fusion. No args needed — uses previous dense and bm25 results automatically.
- filter_relative: Keeps top cases by relative score. No args needed — uses aggregate_rrf results automatically.

METADATA FILTER FIELDS (all available):
- year: e.g. "2023.0"
- judges: e.g. "YAHYA AFRIDI" (handled by bm25_filter only, not ChromaDB)
- case_type: e.g. "C.A", "Crl.A"
- petitioner: e.g. "Dawood Investment Bank" (use in bm25_filter for partial matching)
- respondent: e.g. "Federation of Pakistan" (use in bm25_filter for partial matching)

IMPORTANT CONSTRAINTS:
- ChromaDB metadata_filter supports ONLY: year (e.g. "2023.0"), case_type (e.g. "C.A"), NOT judges
- Judges filtering is handled ONLY by bm25_filter — never put judges in metadata_filter
- bm25_filter is a plain dict e.g. {"judges": "YAHYA AFRIDI", "year": "2023.0"}
- metadata_filter uses ChromaDB format e.g. {"year": {"$eq": "2023.0"}}
- parse_legal_query handles all this automatically — always call it first

CRITICAL RULE ABOUT BM25_FILTER:
- ALWAYS use "PARSED_BM25_FILTER" as the bm25_filter value — never null, never omit it
- parse_legal_query returns bm25_filter which may contain year, case_type, judges, petitioner, respondent
- Even if you think there is no filter, use "PARSED_BM25_FILTER" — let the executor decide if it's null
- The executor resolves "PARSED_BM25_FILTER" automatically from parse_legal_query output


DEFAULT PIPELINE (use this for most queries):
1. parse_legal_query
2. dense_search
3. bm25_search
4. aggregate_rrf
5. filter_relative

WHEN TO DEVIATE:
- Skip bm25_search only if query is purely conceptual with no keywords (rare)
- Add extra metadata filters if you notice specific fields in the query (e.g. a specific year not mentioned explicitly)
- If replanning after failure, try different parameters (different k, different query phrasing)

OUTPUT FORMAT (JSON array only, no explanation):
[
  {"tool": "parse_legal_query", "args": {"user_query": "original query here"}},
  {"tool": "dense_search", "args": {"query": "SEMANTIC_QUERY", "metadata_filter": null, "k": 30}},
  {"tool": "bm25_search", "args": {"query": "SEMANTIC_QUERY", "bm25_filter": null, "k": 30}},
  {"tool": "aggregate_rrf", "args": {}},
  {"tool": "filter_relative", "args": {}}
]

NOTE: After parse_legal_query runs, use "PARSED_SEMANTIC_QUERY", "PARSED_METADATA_FILTER", "PARSED_BM25_FILTER" as placeholder values in subsequent tools — the executor will resolve these from parse_legal_query output automatically.
NOTE ON PLACEHOLDERS:
- use placeholder values like "TOP_RANKED_CASE"
+ use references like "$aggregate_rrf[0].case_id"
- Use "REWRITTEN_QUERY" as query value in tools after rewrite_query runs — executor resolves it automatically
- Use "PARSED_SEMANTIC_QUERY", "PARSED_METADATA_FILTER", "PARSED_BM25_FILTER" after parse_legal_query
- Use "TOP_RANKED_CASE", "SECOND_RANKED_CASE", "THIRD_RANKED_CASE" for expand_case case_id
- bm25_filter must always be a dict like {"judges": "YAHYA AFRIDI"} or null — never a plain string
"""

QA_PLANNER_PROMPT = """You are a Planner for a Pakistani Supreme Court legal QA system.

You will receive a user query and must output a JSON plan — an ordered list of tool calls to find and answer the question.

AVAILABLE TOOLS:
- detect_case_reference: Checks if query mentions a specific case number. Args: query (str). Returns case_id or null.
- parse_legal_query: Parses query into semantic_query, metadata_filter, bm25_filter. Use when no explicit case reference found.
- dense_search: Semantic embedding search. Args: query (str), metadata_filter (dict or null), k (int, default 30)
- bm25_search: Keyword search. Args: query (str), bm25_filter (dict or null), k (int, default 30)
- aggregate_rrf: Combines dense and BM25 results. No args needed.
- expand_case: Retrieves all chunks from a specific case. Args: case_id (str), k (int, default 15). Use "TOP_RANKED_CASE" as case_id to expand the top result from aggregate_rrf automatically.
- generate_answer: Generates answer using Groq/Llama from expanded chunks. No args needed. ALWAYS include this.
- verify_answer: Verifies answer is grounded in chunks using Gemini. No args needed. ALWAYS include this — NEVER skip.
- rewrite_query: Rewrites query with legal synonyms for better retrieval. Args: query (str). Use in plan 2 if plan 1 failed.

METADATA FILTER FIELDS (all available):
- year: e.g. "2023.0"
- judges: e.g. "YAHYA AFRIDI" (handled by bm25_filter only, not ChromaDB)
- case_type: e.g. "C.A", "Crl.A"
- petitioner: e.g. "Dawood Investment Bank" (use in bm25_filter for partial matching)
- respondent: e.g. "Federation of Pakistan" (use in bm25_filter for partial matching)

IMPORTANT CONSTRAINTS:
- ChromaDB metadata_filter supports ONLY: year, case_type — NOT judges
- Judges filtering handled ONLY by bm25_filter
- parse_legal_query handles filter formatting automatically
- verify_answer is MANDATORY in every plan — never omit it
- generate_answer is MANDATORY in every plan — never omit it


CRITICAL RULE ABOUT BM25_FILTER:
- ALWAYS use "PARSED_BM25_FILTER" as the bm25_filter value — never null, never omit it
- parse_legal_query returns bm25_filter which may contain year, case_type, judges, petitioner, respondent
- Even if you think there is no filter, use "PARSED_BM25_FILTER" — let the executor decide if it's null
- The executor resolves "PARSED_BM25_FILTER" automatically from parse_legal_query output

CRITICAL: When detect_case_reference is in the plan, ALWAYS use "DETECTED_CASE_ID" as the case_id in expand_case — never null, never the literal case number. The executor resolves it automatically.

CORRECT:
{"tool": "expand_case", "args": {"case_id": "DETECTED_CASE_ID", "k": 15}}

WRONG:
{"tool": "expand_case", "args": {"case_id": null, "k": 15}}
{"tool": "expand_case", "args": {"case_id": "C.A.875_2017", "k": 15}}


DEFAULT PIPELINE — explicit case reference:
1. detect_case_reference
2. expand_case (with detected case_id)
3. generate_answer
4. verify_answer

DEFAULT PIPELINE — no explicit case reference:
1. detect_case_reference
2. parse_legal_query
3. dense_search
4. bm25_search
5. aggregate_rrf
6. expand_case (with "TOP_RANKED_CASE")
7. generate_answer
8. verify_answer

WHEN TO DEVIATE:
- If query mentions a party name, organization, or plaintiff — add it to bm25_filter as a keyword search
- If replanning after failure: use rewrite_query, try a different ranked case ("SECOND_RANKED_CASE"), or add/remove filters
- If plan 1 failed because wrong case was expanded — try "SECOND_RANKED_CASE" or "THIRD_RANKED_CASE" in plan 2

OUTPUT FORMAT (JSON array only, no explanation):
[
  {"tool": "detect_case_reference", "args": {"query": "original query here"}},
  {"tool": "parse_legal_query", "args": {"user_query": "original query here"}},
  {"tool": "dense_search", "args": {"query": "PARSED_SEMANTIC_QUERY", "metadata_filter": "PARSED_METADATA_FILTER", "k": 30}},
  {"tool": "bm25_search", "args": {"query": "PARSED_SEMANTIC_QUERY", "bm25_filter": "PARSED_BM25_FILTER", "k": 30}},
  {"tool": "aggregate_rrf", "args": {}},
  {"tool": "expand_case", "args": {"case_id": "TOP_RANKED_CASE", "k": 15}},
  {"tool": "generate_answer", "args": {}},
  {"tool": "verify_answer", "args": {}}
]
NOTE ON PLACEHOLDERS:
- Use "REWRITTEN_QUERY" as query value in tools after rewrite_query runs — executor resolves it automatically
- Use "PARSED_SEMANTIC_QUERY", "PARSED_METADATA_FILTER", "PARSED_BM25_FILTER" after parse_legal_query
- Use "TOP_RANKED_CASE", "SECOND_RANKED_CASE", "THIRD_RANKED_CASE" for expand_case case_id
- bm25_filter must always be a dict like {"judges": "YAHYA AFRIDI"} or null — never a plain string
"""

REPLAN_PROMPT = """You are a Planner for a Pakistani Supreme Court legal system. Your first plan failed. Generate a new plan.

ORIGINAL QUERY: {query}

PLAN 1 THAT FAILED:
{plan_1}

FAILURE REASON:
{failure_reason}

RESULTS FROM PLAN 1:
{plan_1_results}

STRICT PLACEHOLDER RULES — only these placeholders are valid:
- "PARSED_SEMANTIC_QUERY" — query from parse_legal_query
- "PARSED_METADATA_FILTER" — chroma filter from parse_legal_query
- "PARSED_BM25_FILTER" — bm25 filter from parse_legal_query
- "REWRITTEN_QUERY" — output from rewrite_query
- "DETECTED_CASE_ID" — output from detect_case_reference
- "TOP_RANKED_CASE" — top case from aggregate_rrf
- "SECOND_RANKED_CASE" — second case from aggregate_rrf
- "THIRD_RANKED_CASE" — third case from aggregate_rrf
DO NOT invent any other placeholder names.
DO NOT pass arguments to aggregate_rrf — it takes no arguments.
DO NOT call aggregate_rrf without first calling both dense_search and bm25_search.

STRATEGIES FOR PLAN 2:
- If failure was "no chunks found for X" → case may be stored with SC-PK_ prefix, try "SC-PK_X" directly in expand_case
- If failure was "generate_answer returned Context insufficient" → wrong case expanded, try SECOND_RANKED_CASE or run full search pipeline first
- If failure was "verify_answer returned NO" → wrong case, try SECOND_RANKED_CASE
- If failure was token/size error → keep same case, use k=6 in expand_case
- If no case reference detected → run full search: parse_legal_query → dense_search → bm25_search → aggregate_rrf → expand_case(TOP_RANKED_CASE)
- If replanning after rewrite → use "REWRITTEN_QUERY" as query in dense_search and bm25_search, NOT as user_query in parse_legal_query

AVAILABLE TOOLS: detect_case_reference, parse_legal_query, dense_search, bm25_search, aggregate_rrf, expand_case, generate_answer, verify_answer, rewrite_query

CONSTRAINTS:
- verify_answer and generate_answer are MANDATORY
- aggregate_rrf takes NO arguments — always: {{"tool": "aggregate_rrf", "args": {{}}}}
- bm25_filter must be a dict or "PARSED_BM25_FILTER" — never a plain string

Return a JSON object with key 'plan' containing the array of tool calls.
"""

In [24]:
import json
import re
from typing import TypedDict, Literal, Optional
from langgraph.graph import StateGraph, END

ROUTER_PROMPT = """You are a Router deciding which agent handles the query.

Route to CASE_SEARCH if:
- User wants to FIND/SEARCH/LOCATE cases
- User asks for LIST of cases
- Query contains: "find", "search", "show me", "cases about", "cases by", "cases from"
- Examples: "Find cases about Section 302", "Show me tax cases from 2023", "Cases by Justice Yahya"

Route to QA if:
- User asks a QUESTION requiring an answer
- User wants EXPLANATION/INTERPRETATION
- Query contains: "what", "why", "how", "did", "was", "explain"
- Examples: "What does Section 302 cover?", "What was the court's ruling in C.A.142/2019?"

AMBIGUOUS CASES:
- "Cases about X" → CASE_SEARCH
- "What cases discuss X" → CASE_SEARCH
- "What did court say about X" → QA

Respond with ONLY JSON:
{"agent": "case_search"} or {"agent": "qa"}
"""

# ============================================
# STATE
# ============================================

class AgenticState(TypedDict):
    query: str
    agent_type: str
    plan: list
    plan_results: dict
    plan_number: int
    failure_reason: str
    final_result: str

# ============================================
# PLANNER
# ============================================

def _call_planner(prompt: str) -> list:
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system", 
                "content": "You are a planning assistant. Always respond with a valid JSON object containing a single key 'plan' whose value is an array of tool call objects."
            },
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    result = json.loads(response.choices[0].message.content)
    # Extract the plan array from the wrapper object
    if "plan" in result:
        return result["plan"]
    # Fallback — if model returned array directly inside another key, find it
    for v in result.values():
        if isinstance(v, list):
            return v
    return []

def _generate_plan(query: str, agent_type: str) -> list:
    if agent_type == "case_search":
        prompt = f"{CASE_SEARCH_PLANNER_PROMPT}\n\nQuery: {query}\n\nReturn a JSON object with key 'plan' containing the array of tool calls."
    else:
        prompt = f"{QA_PLANNER_PROMPT}\n\nQuery: {query}\n\nReturn a JSON object with key 'plan' containing the array of tool calls."
    plan = _call_planner(prompt)
    print(f"  📋 Plan generated: {[s['tool'] for s in plan]}")
    return plan

def _generate_replan(query: str, plan_1: list, failure_reason: str, plan_1_results: dict) -> list:
    prompt = REPLAN_PROMPT.format(
        query=query,
        plan_1=json.dumps(plan_1, indent=2),
        failure_reason=failure_reason,
        plan_1_results=json.dumps(
            {k: str(v)[:300] for k, v in plan_1_results.items()},
            indent=2
        )
    )
    prompt += "\n\nReturn a JSON object with key 'plan' containing the array of tool calls."
    plan = _call_planner(prompt)
    print(f"  📋 Replan generated: {[s['tool'] for s in plan]}")
    return plan

def route_query_agentic(state: AgenticState) -> AgenticState:
    query = state["query"]
    
    system = """You are a Router deciding which agent handles a legal query.

Route to case_search if:
- User wants to FIND/SEARCH/LOCATE cases
- User asks for LIST of cases
- Query contains: find, search, show me, cases about, cases by, cases from

Route to qa if:
- User asks a QUESTION requiring an answer
- User wants EXPLANATION/INTERPRETATION
- Query contains: what, why, how, did, was, explain

AMBIGUOUS CASES:
- Cases about X → case_search
- What cases discuss X → case_search
- What did court say about X → qa

Return ONLY a JSON object with key 'agent' and value either 'case_search' or 'qa'."""

    try:
        result = _call_groq_json(f"Query: {query}", system=system)
        agent_choice = result.get("agent", "qa")
    except:
        agent_choice = "qa"

    print(f"🔀 Router: {agent_choice}")
    state["agent_type"] = agent_choice
    return state
# ============================================
# EXECUTOR
# ============================================

def _resolve_arg(value, results: dict):
    if value == "PARSED_SEMANTIC_QUERY":
        return results.get("parse_legal_query", {}).get("semantic_query", "")
    if value == "PARSED_METADATA_FILTER":
        val = results.get("parse_legal_query", {}).get("metadata_filter", None)
        print(f"  resolved PARSED_METADATA_FILTER: {val}")
        return val
    if value == "PARSED_BM25_FILTER":
        return results.get("parse_legal_query", {}).get("bm25_filter", None)
    if value == "REWRITTEN_QUERY":  # ← add this
        return results.get("rewrite_query", "")
    if value == "TOP_RANKED_CASE":
        ranked = results.get("aggregate_rrf", [])
        return ranked[0]["case_id"] if ranked else None
    if value == "SECOND_RANKED_CASE":
        ranked = results.get("aggregate_rrf", [])
        return ranked[1]["case_id"] if len(ranked) > 1 else None
    if value == "THIRD_RANKED_CASE":
        ranked = results.get("aggregate_rrf", [])
        return ranked[2]["case_id"] if len(ranked) > 2 else None
    if value == "DETECTED_CASE_ID":
        return results.get("detect_case_reference")
        
    return value

def _resolve_args(args: dict, results: dict) -> dict:
    return {k: _resolve_arg(v, results) for k, v in args.items()}

def _execute_plan(plan: list, query: str) -> tuple[dict, str]:
    results = {}
    failure_reason = None

    for step in plan:
        tool = step["tool"]
        raw_args = step.get("args", {})
        args = _resolve_args(raw_args, results)

        print(f"  ⚙️ Executing: {tool}({args})")

        try:
            if tool == "parse_legal_query":
                results["parse_legal_query"] = _parse_legal_query(args.get("user_query", query))

            elif tool == "detect_case_reference":
                results["detect_case_reference"] = _detect_case_reference(args.get("query", query))

            elif tool == "dense_search":
                results["dense_search"] = _dense_search(
                    args.get("query", query),
                    args.get("metadata_filter"),
                    args.get("k", 30)
                )

            elif tool == "bm25_search":
                results["bm25_search"] = _bm25_search_filtered(
                    args.get("query", query),
                    args.get("bm25_filter"),
                    args.get("k", 30)
                )

            elif tool == "aggregate_rrf":
                dense = results.get("dense_search", [])
                bm25 = results.get("bm25_search", [])
                results["aggregate_rrf"] = _aggregate_rrf(dense, bm25)
                if not results["aggregate_rrf"]:
                    failure_reason = "aggregate_rrf returned no results"
                    break

            elif tool == "filter_relative":
                ranked = results.get("aggregate_rrf", [])
                results["filter_relative"] = _filter_relative(ranked)

            elif tool == "expand_case":
                case_id = args.get("case_id")
                if not case_id:
                    failure_reason = "expand_case: no case_id resolved"
                    break
                chunks = _expand_case(case_id, args.get("k", 15))
                results["expand_case"] = chunks
                results["expanded_case_id"] = case_id
                if not chunks:
                    failure_reason = f"expand_case: no chunks found for {case_id}"
                    break

            elif tool == "rewrite_query":
                results["rewrite_query"] = _rewrite_query(args.get("query", query))

            elif tool == "generate_answer":
                chunks = results.get("expand_case", [])
                if not chunks:
                    failure_reason = "generate_answer: no chunks available"
                    break
                effective_query = results.get("rewrite_query", query)
                answer = _generate_answer(effective_query, chunks)
                results["generate_answer"] = answer
                if not answer or "Context insufficient" in answer:
                    failure_reason = f"generate_answer returned: {answer[:100]}"
                    break

            elif tool == "verify_answer":
                answer = results.get("generate_answer", "")
                chunks = results.get("expand_case", [])
                if not answer or not chunks:
                    failure_reason = "verify_answer: missing answer or chunks"
                    break
                effective_query = results.get("rewrite_query", query)
                verified = _verify_with_gemini(effective_query, answer, chunks)
                results["verify_answer"] = verified
                if not verified:
                    failure_reason = "verify_answer returned NO — answer not grounded in chunks"
                    break

        except Exception as e:
            failure_reason = f"{tool} raised exception: {str(e)}"
            print(f"  ❌ {failure_reason}")
            break

    return results, failure_reason

# ============================================
# AGENT NODES
# ============================================

def case_search_agent_node(state: AgenticState) -> AgenticState:
    query = state["query"]
    print(f"\n🔍 CASE SEARCH AGENT executing...")

    # Plan 1
    print("\n📋 Generating Plan 1...")
    plan = _generate_plan(query, "case_search")
    results, failure_reason = _execute_plan(plan, query)

    if not failure_reason:
        filtered = results.get("filter_relative", results.get("aggregate_rrf", []))
        output = "Found Cases:\n\n"
        for i, case in enumerate(filtered, 1):
            output += f"{i}. Case ID: {case['case_id']}\n"
            output += f"   Score: {case['score']}\n"
            output += f"   Methods: {', '.join(case['matched_on'])}\n"
            output += f"   Preview: {case['preview'][:150]}\n\n"
        state["final_result"] = output
        return state

    # Plan 2
    print(f"\n⚠️ Plan 1 failed: {failure_reason}")
    print("\n📋 Generating Plan 2...")
    plan_2 = _generate_replan(query, plan, failure_reason, results)
    results_2, failure_reason_2 = _execute_plan(plan_2, query)

    if not failure_reason_2:
        filtered = results_2.get("filter_relative", results_2.get("aggregate_rrf", []))
        output = "Found Cases:\n\n"
        for i, case in enumerate(filtered, 1):
            output += f"{i}. Case ID: {case['case_id']}\n"
            output += f"   Score: {case['score']}\n"
            output += f"   Methods: {', '.join(case['matched_on'])}\n"
            output += f"   Preview: {case['preview'][:150]}\n\n"
        state["final_result"] = output
        return state

    state["final_result"] = "Could not find relevant cases."
    return state


def qa_agent_node(state: AgenticState) -> AgenticState:
    query = state["query"]
    print(f"\n💬 QA AGENT executing...")

    # Plan 1
    print("\n📋 Generating Plan 1...")
    plan = _generate_plan(query, "qa")
    results, failure_reason = _execute_plan(plan, query)

    if not failure_reason:
        state["final_result"] = results.get("generate_answer", "Context insufficient.")
        return state

    # Plan 2
    print(f"\n⚠️ Plan 1 failed: {failure_reason}")
    print("\n📋 Generating Plan 2...")
    plan_2 = _generate_replan(query, plan, failure_reason, results)
    results_2, failure_reason_2 = _execute_plan(plan_2, query)

    if not failure_reason_2:
        state["final_result"] = results_2.get("generate_answer", "Context insufficient.")
        return state

    state["final_result"] = "Context insufficient. Answer could not be verified after replanning."
    return state

# ============================================
# ROUTER NODE
# ============================================

def route_query_agentic(state: AgenticState) -> AgenticState:
    query = state["query"]
    response = gemini_model.generate_content(
        f"{ROUTER_PROMPT}\n\nQuery: {query}",
        generation_config={"temperature": 0}
    )
    try:
        result = json.loads(re.sub(r"```json|```", "", response.text).strip())
        agent_choice = result.get("agent", "qa")
    except:
        agent_choice = "qa"

    print(f"🔀 Router: {agent_choice}")
    state["agent_type"] = agent_choice
    return state

def decide_agent(state: AgenticState) -> Literal["case_search_agent", "qa_agent"]:
    return "case_search_agent" if state["agent_type"] == "case_search" else "qa_agent"

# ============================================
# GRAPH
# ============================================

def build_agentic_graph():
    workflow = StateGraph(AgenticState)
    workflow.add_node("route", route_query_agentic)
    workflow.add_node("case_search_agent", case_search_agent_node)
    workflow.add_node("qa_agent", qa_agent_node)
    workflow.set_entry_point("route")
    workflow.add_conditional_edges(
        "route",
        decide_agent,
        {
            "case_search_agent": "case_search_agent",
            "qa_agent": "qa_agent"
        }
    )
    workflow.add_edge("case_search_agent", END)
    workflow.add_edge("qa_agent", END)
    return workflow.compile()

def run_agentic_system(query: str, verbose: bool = True):
    graph = build_agentic_graph()
    result = graph.invoke({
        "query": query,
        "agent_type": "",
        "plan": [],
        "plan_results": {},
        "plan_number": 1,
        "failure_reason": "",
        "final_result": ""
    })

    if verbose:
        print("\n" + "=" * 60)
        print(f"FINAL RESULT:\n{result['final_result']}")

    return result["final_result"]

print("✅ Agentic system ready")

✅ Agentic system ready


# Step 6: Test


In [73]:
print(run_agentic_system("Find cases by Justice Yahya Afridi"))

🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'Find cases by Justice Yahya Afridi'})
  🔍 Parsed intent: {'semantic_query': 'Justice Yahya Afridi', 'metadata_filter': {'judges': 'MR. JUSTICE YAHYA AFRIDI'}}
  chroma items: []
  normalized_filter: {'judges': 'YAHYA AFRIDI'}
  chroma_filter: None
  bm25_filter: {'judges': 'YAHYA AFRIDI'}
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'Justice Yahya Afridi', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'Justice Yahya Afridi', 'bm25_filter': {'judges': 'YAHYA AFRIDI'}, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: C.P.L.A.3155-L_2023
   Score: 0.0315
   Methods: DENSE, BM25
   Preview: IN THE SUPREME COURT OF PAKISTAN
(

In [74]:
result = run_agentic_system("Find cases presided by Justice Munib Akhtar", verbose=True)

# Split by case
cases = result.split("\n\n")[1:]  # Skip header
for case in cases:
    if case.strip():
        print(case)
        print("-"*60)

🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'Find cases presided by Justice Munib Akhtar'})
  🔍 Parsed intent: {'semantic_query': 'Justice Munib Akhtar', 'metadata_filter': {'judges': 'MR. JUSTICE MUNIB AKHTAR'}}
  chroma items: []
  normalized_filter: {'judges': 'MUNIB AKHTAR'}
  chroma_filter: None
  bm25_filter: {'judges': 'MUNIB AKHTAR'}
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'Justice Munib Akhtar', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'Justice Munib Akhtar', 'bm25_filter': {'judges': 'MUNIB AKHTAR'}, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: C.P.L.A.185_2024
   Score: 0.0311
   Methods: DENSE, BM25
   Preview: IN THE SUPREME COURT OF PAKI

In [27]:
result = run_agentic_system("I need child custody cases from 2022", verbose=True)

🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'child custody cases from 2022'})
  🔍 Parsed intent: {'semantic_query': 'child custody family law guardianship parental rights', 'metadata_filter': {'year': '2022.0'}}
  normalized_filter: {'year': '2022.0'}
  chroma_filter: {'year': {'$eq': '2022.0'}}
  bm25_filter: {'year': '2022.0'}
  resolved PARSED_METADATA_FILTER: {'year': {'$eq': '2022.0'}}
  ⚙️ Executing: dense_search({'query': 'child custody family law guardianship parental rights', 'metadata_filter': {'year': {'$eq': '2022.0'}}, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'child custody family law guardianship parental rights', 'bm25_filter': {'year': '2022.0'}, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: C.P.L.A

In [39]:
result = run_agentic_system("Find me cases about child custody and guardianship disputes", verbose=True)

🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'child custody and guardianship disputes'})
  🔍 Parsed intent: {'semantic_query': 'child custody guardianship disputes family law parental rights visitation', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'child custody guardianship disputes family law parental rights visitation', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'child custody guardianship disputes family law parental rights visitation', 'bm25_filter': None, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: C.R.P.458_2024
   Score: 0.032
   Methods: DENSE, BM25
   Prev

In [69]:
result = run_agentic_system("Article 62 disqualification cases", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'Article 62 disqualification cases'})
  🔍 Parsed intent: {'semantic_query': 'Article 62 disqualification ineligibility election candidacy public office', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'Article 62 disqualification ineligibility election candidacy public office', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'Article 62 disqualification ineligibility election candidacy public office', 'bm25_filter': None, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: C.A.981_2018
   Score: 0.0325
   Methods: DENSE, BM25
   Preview:

In [70]:
result = run_agentic_system("I need cases about Civil appeals about property disputes", verbose=True)

🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'I need cases about Civil appeals about property disputes'})
  🔍 Parsed intent: {'semantic_query': 'civil appeal property dispute ownership rights inheritance', 'metadata_filter': {'case_type': 'C.A'}}
  chroma items: [('case_type', 'C.A')]
  normalized_filter: {'case_type': 'C.A'}
  chroma_filter: {'case_type': {'$eq': 'C.A'}}
  bm25_filter: {'case_type': 'C.A'}
  resolved PARSED_METADATA_FILTER: {'case_type': {'$eq': 'C.A'}}
  ⚙️ Executing: dense_search({'query': 'civil appeal property dispute ownership rights inheritance', 'metadata_filter': {'case_type': {'$eq': 'C.A'}}, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'civil appeal property dispute ownership rights inheritance', 'bm25_filter': {'case_type': 'C.A'}, 'k': 30})
  ⚙️ Executing: aggrega

In [71]:
result = run_agentic_system("I need cases about employment termination and wrongful dismissal", verbose=True)

🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'I need cases about employment termination and wrongful dismissal'})
  🔍 Parsed intent: {'semantic_query': 'employment termination wrongful dismissal retrenchment layoff job loss compensation severance package', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'employment termination wrongful dismissal retrenchment layoff job loss compensation severance package', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'employment termination wrongful dismissal retrenchment layoff job loss compensation severance package', 'bm25_filter': None, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relativ

In [72]:
result = run_agentic_system("Section 302 PPC murder cases", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'Section 302 PPC murder cases'})
  🔍 Parsed intent: {'semantic_query': 'Section 302 PPC murder qatl-i-amd homicide conviction', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'Section 302 PPC murder qatl-i-amd homicide conviction', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'Section 302 PPC murder qatl-i-amd homicide conviction', 'bm25_filter': None, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: Crl.A.530_2022
   Score: 0.0306
   Methods: DENSE, BM25
   Preview: raised before us is whether the High Court was justified in enhan

In [75]:
result = run_agentic_system("Recent tax cases from 2023", verbose=True)

🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'Recent tax cases from 2023'})
  🔍 Parsed intent: {'semantic_query': 'tax taxation revenue FBR customs', 'metadata_filter': {'year': '2023.0'}}
  chroma items: [('year', '2023.0')]
  normalized_filter: {'year': '2023.0'}
  chroma_filter: {'year': {'$eq': '2023.0'}}
  bm25_filter: {'year': '2023.0'}
  resolved PARSED_METADATA_FILTER: {'year': {'$eq': '2023.0'}}
  ⚙️ Executing: dense_search({'query': 'tax taxation revenue FBR customs', 'metadata_filter': {'year': {'$eq': '2023.0'}}, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'tax taxation revenue FBR customs', 'bm25_filter': {'year': '2023.0'}, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: C.P.L.A.824-K_2023
   Score: 0.0328


In [25]:
result = run_agentic_system("Find criminal appeals regarding murder from 2019 presided by Justice Yahya", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'Find criminal appeals regarding murder from 2019 presided by Justice Yahya'})
  🔍 Parsed intent: {'semantic_query': 'criminal appeal murder qatl-i-amd homicide conviction', 'metadata_filter': {'year': '2019.0', 'judges': 'MR. JUSTICE YAHYA AFRIDI', 'case_type': 'Crl.A'}}
  normalized_filter: {'year': '2019.0', 'judges': 'YAHYA AFRIDI', 'case_type': 'Crl.A'}
  chroma_filter: {'$and': [{'year': {'$eq': '2019.0'}}, {'case_type': {'$eq': 'Crl.A'}}]}
  bm25_filter: {'year': '2019.0', 'judges': 'YAHYA AFRIDI', 'case_type': 'Crl.A'}
  resolved PARSED_METADATA_FILTER: {'$and': [{'year': {'$eq': '2019.0'}}, {'case_type': {'$eq': 'Crl.A'}}]}
  ⚙️ Executing: dense_search({'query': 'criminal appeal murder qatl-i-amd homicide conviction', 'metadata_filter': {'$and

In [26]:
result = run_agentic_system("cases involving First Dawood Investment Bank", verbose=True)

🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'cases involving First Dawood Investment Bank'})
  🔍 Parsed intent: {'semantic_query': 'First Dawood Investment Bank financial institution case', 'metadata_filter': {'petitioner': 'First Dawood Investment Bank'}}
  normalized_filter: {'petitioner': 'First Dawood Investment Bank'}
  chroma_filter: None
  bm25_filter: {'petitioner': 'First Dawood Investment Bank'}
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'First Dawood Investment Bank financial institution case', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'First Dawood Investment Bank financial institution case', 'bm25_filter': {'petitioner': 'First Dawood Investment Bank'}, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filt

# Now need to check QA RAG

In [66]:
result = run_agentic_system("In the case involving Frontier Holdings Limited, what judicial policy informed the Supreme Court's decision to maintain the interim restraining order during the appeals process?", verbose=True)

# Split by case
cases = result.split("\n\n")[1:]  # Skip header
for case in cases:
    if case.strip():
        print(case)
        print("-"*60)

🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': "In the case involving Frontier Holdings Limited, what judicial policy informed the Supreme Court's decision to maintain the interim restraining order during the appeals process?"})
  ⚙️ Executing: parse_legal_query({'user_query': "In the case involving Frontier Holdings Limited, what judicial policy informed the Supreme Court's decision to maintain the interim restraining order during the appeals process?"})
  🔍 Parsed intent: {'semantic_query': 'judicial policy interim restraining order appeals process Supreme Court decision Frontier Holdings Limited', 'metadata_filter': {'petitioner': 'Frontier Holdings Limited'}}
  chroma items: []
  normalized_filter: {'petitioner': 'Frontier Holdings Limited'}
  chroma_filter

In [67]:
result = run_agentic_system("According to the Supreme Court's interpretation of the Muslim Family Laws Ordinance, 1961, are the descendants beyond a grandchild entitled to an inheritance share under Section 4?", verbose=True)

# Split by case
cases = result.split("\n\n")[1:]  # Skip header
for case in cases:
    if case.strip():
        print(case)
        print("-"*60)

🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': "According to the Supreme Court's interpretation of the Muslim Family Laws Ordinance, 1961, are the descendants beyond a grandchild entitled to an inheritance share under Section 4?"})
  ⚙️ Executing: parse_legal_query({'user_query': "According to the Supreme Court's interpretation of the Muslim Family Laws Ordinance, 1961, are the descendants beyond a grandchild entitled to an inheritance share under Section 4?"})
  🔍 Parsed intent: {'semantic_query': 'Muslim Family Laws Ordinance 1961 Section 4 inheritance share grandchildren descendants Supreme Court interpretation', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing

In [68]:
result = run_agentic_system("What was the Supreme Court's final decision regarding the power of the Election Commission of Pakistan to conduct a new election in a constituency (NA-91 Sargodha-IV) after the final results had been formally announced?", verbose=True)

# Split by case
cases = result.split("\n\n")[1:]  # Skip header
for case in cases:
    if case.strip():
        print(case)
        print("-"*60)

🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': "What was the Supreme Court's final decision regarding the power of the Election Commission of Pakistan to conduct a new election in a constituency (NA-91 Sargodha-IV) after the final results had been formally announced?"})
  ⚙️ Executing: parse_legal_query({'user_query': "What was the Supreme Court's final decision regarding the power of the Election Commission of Pakistan to conduct a new election in a constituency (NA-91 Sargodha-IV) after the final results had been formally announced?"})
  🔍 Parsed intent: {'semantic_query': 'Election Commission of Pakistan ECP power to conduct new election in constituency NA-91 Sargodha-IV after final results announcement Supreme Court decision', 'metadata_filter': {'case_type

ok

In [65]:
result = run_agentic_system("Under the Income Tax Ordinance, what action must the Appellate Tribunal Inland Revenue take if a party fails to appear at a hearing, according to the Supreme Court's ruling?", verbose=True)

# Split by case
cases = result.split("\n\n")[1:]  # Skip header
for case in cases:
    if case.strip():
        print(case)
        print("-"*60)

🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': "Under the Income Tax Ordinance, what action must the Appellate Tribunal Inland Revenue take if a party fails to appear at a hearing, according to the Supreme Court's ruling?"})
  ⚙️ Executing: parse_legal_query({'user_query': "Under the Income Tax Ordinance, what action must the Appellate Tribunal Inland Revenue take if a party fails to appear at a hearing, according to the Supreme Court's ruling?"})
  🔍 Parsed intent: {'semantic_query': 'Income Tax Ordinance Appellate Tribunal Inland Revenue hearing absence Supreme Court ruling decision', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'Inco

In [28]:
result = run_agentic_system("What was the primary procedural direction given to the Islamabad High Court after the Supreme Court set aside its order in the case involving Justice Tariq Mehmood Jahangiri?", verbose=True)

# Split by case
cases = result.split("\n\n")[1:]  # Skip header
for case in cases:
    if case.strip():
        print(case)
        print("-"*60)

🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'What was the primary procedural direction given to the Islamabad High Court after the Supreme Court set aside its order in the case involving Justice Tariq Mehmood Jahangiri?'})
  ⚙️ Executing: parse_legal_query({'user_query': 'What was the primary procedural direction given to the Islamabad High Court after the Supreme Court set aside its order in the case involving Justice Tariq Mehmood Jahangiri?'})
  🔍 Parsed intent: {'semantic_query': 'Supreme Court set aside order Islamabad High Court procedural direction Justice Tariq Mehmood Jahangiri', 'metadata_filter': {'judges': 'JUSTICE TARIQ MEHMOOD JAHANGIRI'}}
  normalized_filter: {'judges': 'TARIQ MEHMOOD JAHANGIRI'}
  chroma_filter: None
  bm25_filter: {'judges':

## NOT HALLUCINATING

In [29]:
result = run_agentic_system("On what grounds did the Supreme Court overturn the conviction of Sardar Khan and Amjad Ali and grant them an acquittal in their jail petition?", verbose=True)

🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'On what grounds did the Supreme Court overturn the conviction of Sardar Khan and Amjad Ali and grant them an acquittal in their jail petition?'})
  ⚙️ Executing: parse_legal_query({'user_query': 'On what grounds did the Supreme Court overturn the conviction of Sardar Khan and Amjad Ali and grant them an acquittal in their jail petition?'})
  🔍 Parsed intent: {'semantic_query': 'Supreme Court overturn conviction Sardar Khan Amjad Ali acquittal jail petition grounds', 'metadata_filter': {'petitioner': 'Sardar Khan', 'case_type': 'C.P'}}
  normalized_filter: {'petitioner': 'Sardar Khan', 'case_type': 'C.P'}
  chroma_filter: {'case_type': {'$eq': 'C.P'}}
  bm25_filter: {'petitioner': 'Sardar Khan', 'case_type': 'C.P'}

In [30]:
result = run_agentic_system("What legal principle did the Supreme Court invoke when denying petitioners the right to appear in an examination based on the argument that other ineligible people were previously allowed?", verbose=True)

# Split by case
cases = result.split("\n\n")[1:]  # Skip header
for case in cases:
    if case.strip():
        print(case)
        print("-"*60)

🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'What legal principle did the Supreme Court invoke when denying petitioners the right to appear in an examination based on the argument that other ineligible people were previously allowed?'})
  ⚙️ Executing: parse_legal_query({'user_query': 'What legal principle did the Supreme Court invoke when denying petitioners the right to appear in an examination based on the argument that other ineligible people were previously allowed?'})
  🔍 Parsed intent: {'semantic_query': 'legal principle Supreme Court examination eligibility precedent discrimination unequal treatment', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: de

ok

In [36]:
result = run_agentic_system("In the context of the Civil Appeal No. 875 of 2017, why did the Supreme Court allow the appeal concerning the Redemption and Restitution of Mortgaged Lands Act, 1964?", verbose=True)

# Split by case
cases = result.split("\n\n")[1:]  # Skip header
for case in cases:
    if case.strip():
        print(case)
        print("-"*60)

🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'In the context of the Civil Appeal No. 875 of 2017, why did the Supreme Court allow the appeal concerning the Redemption and Restitution of Mortgaged Lands Act, 1964?'})


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


  ⚙️ Executing: expand_case({'case_id': 'C.A.875_2017', 'k': 15})
✅ DB loaded: 9226 chunks
  🔧 Resolved case_id: C.A.875_2017 → SC-PK_C.A.875_2017
  ⚙️ Executing: generate_answer({})
  ⚙️ Executing: verify_answer({})
  ✅ VERDICT: YES

FINAL RESULT:
Selected Case: SC-PK_C.A.875_2017

Answer: The Supreme Court allowed the appeal because it found that the learned High Court had erred in its decision. The contesting respondents' application under the Redemption and Restitution of Mortgaged Lands Act, 1964, was barred by time, and the Board of Revenue had no power to extend the period of limitation under the Act. Furthermore, the second application filed by the contesting respondents was also barred under section 8 of the Act, as no suit for redemption was filed.

Case Id: SC-PK_C.A.875_2017

Relevant chunks: "We invited learned counsel for the contesting respondents to show us the provision of law under which the Board had the power to extend the period of limitation under the Act and, eve

In [38]:
result = run_agentic_system("Does the Supreme Court of Pakistan have appellate jurisdiction over decisions made by the Khyber Pakhtunkhwa Service Tribunal under the relevant Constitutional Article?", verbose=True)

# Split by case
cases = result.split("\n\n")[1:]  # Skip header
for case in cases:
    if case.strip():
        print(case)
        print("-"*60)

🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'Does the Supreme Court of Pakistan have appellate jurisdiction over decisions made by the Khyber Pakhtunkhwa Service Tribunal under the relevant Constitutional Article?'})
  ⚙️ Executing: parse_legal_query({'user_query': 'Does the Supreme Court of Pakistan have appellate jurisdiction over decisions made by the Khyber Pakhtunkhwa Service Tribunal under the relevant Constitutional Article?'})
  🔍 Parsed intent: {'semantic_query': 'Supreme Court of Pakistan appellate jurisdiction Khyber Pakhtunkhwa Service Tribunal Constitutional Article appeal judicial review', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_se

ok


In [37]:
result = run_agentic_system("In C.M.A. 5777 of 2021 and Civil Petition No. 4944 of 2021, what was the legal controversy regarding MDCAT requirements for private medical college admissions?", verbose=True)


🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'In C.M.A. 5777 of 2021 and Civil Petition No. 4944 of 2021, what was the legal controversy regarding MDCAT requirements for private medical college admissions?'})
  ⚙️ Executing: expand_case({'case_id': 'C.M.A.5777_2021', 'k': 15})
  ⚙️ Executing: generate_answer({})
  ⚙️ Executing: verify_answer({})
  ✅ VERDICT: YES

FINAL RESULT:
Selected Case: C.M.A.5777_2021

Answer: The legal controversy revolved around whether private medical colleges could dispense with the requirement of MDCAT (Medical and Dental Colleges Admission Test) for admissions, with the petitioners arguing that the prospectus of the private medical college did not require MDCAT, while the respondents argued that the statutory requirement of MDCAT under the Pakistan Medical Commission Act, 2020, was mandatory for al

In [40]:
result = run_agentic_system("In a narcotics case involving multiple packets and slabs, why is the method of sampling important for conviction?", verbose=True)


🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'In a narcotics case involving multiple packets and slabs, why is the method of sampling important for conviction?'})
  ⚙️ Executing: parse_legal_query({'user_query': 'In a narcotics case involving multiple packets and slabs, why is the method of sampling important for conviction?'})
  🔍 Parsed intent: {'semantic_query': 'narcotics case sampling method conviction multiple packets slabs forensic evidence', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'narcotics case sampling method conviction multiple packets slabs forensic evidence', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_s

In [46]:
result = run_agentic_system("What was the Supreme Court's conclusion regarding the validity of the delegation of powers by PEMRA to its Chairman in the case involving the Pakistan Broadcasters Association?", verbose=True)


🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': "What was the Supreme Court's conclusion regarding the validity of the delegation of powers by PEMRA to its Chairman in the case involving the Pakistan Broadcasters Association?"})
  ⚙️ Executing: parse_legal_query({'user_query': "What was the Supreme Court's conclusion regarding the validity of the delegation of powers by PEMRA to its Chairman in the case involving the Pakistan Broadcasters Association?"})
  🔍 Parsed intent: {'semantic_query': 'Supreme Court PEMRA delegation of powers Chairman Pakistan Broadcasters Association validity', 'metadata_filter': {'petitioner': 'Pakistan Broadcasters Association'}}
  chroma items: []
  normalized_filter: {'petitioner': 'Pakistan Broadcasters Association'}
  chroma_filter

In [47]:
result = run_agentic_system("In the case of Konish Enterprise (Pvt) Ltd. vs. PTCL, what was the specific point on which the Supreme Court originally granted leave to appeal to PTCL?", verbose=True)


🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'In the case of Konish Enterprise (Pvt) Ltd. vs. PTCL, what was the specific point on which the Supreme Court originally granted leave to appeal to PTCL?'})
  ⚙️ Executing: expand_case({'case_id': None, 'k': 15})

⚠️ Plan 1 failed: expand_case: no case_id resolved

📋 Generating Plan 2...
  📋 Replan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: parse_legal_query({'query': 'In the case of Konish Enterprise (Pvt) Ltd. vs. PTCL, what was the specific point on which the Supreme Court originally granted leave to appeal to PTCL?'})
  🔍 Parsed intent: {'semantic_query': 'Konish Enterprise Pvt Ltd vs PTCL Supreme Court leave to appeal PTCL grounds', 'metadata_filter': {'petitioner': 'Konish 

In [48]:
result = run_agentic_system("Why did the Supreme Court refuse leave to appeal to the heirs of Abdul Shakoor in their property dispute against the heirs of Muhammad Hanif?", verbose=True)


🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'Why did the Supreme Court refuse leave to appeal to the heirs of Abdul Shakoor in their property dispute against the heirs of Muhammad Hanif?'})
  ⚙️ Executing: parse_legal_query({'user_query': 'Why did the Supreme Court refuse leave to appeal to the heirs of Abdul Shakoor in their property dispute against the heirs of Muhammad Hanif?'})
  🔍 Parsed intent: {'semantic_query': 'Supreme Court leave to appeal refusal property dispute inheritance heirs Abdul Shakoor Muhammad Hanif', 'metadata_filter': {'petitioner': 'Abdul Shakoor', 'respondent': 'Muhammad Hanif'}}
  chroma items: []
  normalized_filter: {'petitioner': 'Abdul Shakoor', 'respondent': 'Muhammad Hanif'}
  chroma_filter: None
  bm25_filter: {'petitioner': 

# EDGE CASES

In [49]:
result = run_agentic_system("What are the landmark cases on Article 10A?", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'What are the landmark cases on Article 10A'})
  🔍 Parsed intent: {'semantic_query': 'Article 10A fundamental rights fair trial due process landmark cases', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'Article 10A fundamental rights fair trial due process landmark cases', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'Article 10A fundamental rights fair trial due process landmark cases', 'bm25_filter': None, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: Crl.P.L.A.636_2022
   Score: 0.0328
   Methods: DENSE, BM25
   Preview: Hi

In [50]:
result = run_agentic_system("Section 302 qatl cases", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'Section 302 qatl cases'})
  🔍 Parsed intent: {'semantic_query': 'Section 302 PPC qatl-i-amd murder homicide conviction', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'Section 302 PPC qatl-i-amd murder homicide conviction', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'Section 302 PPC qatl-i-amd murder homicide conviction', 'bm25_filter': None, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: Crl.P.L.A.344_2018
   Score: 0.03
   Methods: DENSE, BM25
   Preview: sudden provocation as to the cases falling under clause ( c ) of sect

In [51]:
result = run_agentic_system("Crl.P.L.A.645-L_2025", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'Crl.P.L.A.645-L_2025'})
  🔍 Parsed intent: {'semantic_query': 'Crl.P.L.A.645-L_2025 criminal law appeal', 'metadata_filter': {'year': '2025.0', 'case_type': 'Crl.A'}}
  chroma items: [('year', '2025.0'), ('case_type', 'Crl.A')]
  normalized_filter: {'year': '2025.0', 'case_type': 'Crl.A'}
  chroma_filter: {'$and': [{'year': {'$eq': '2025.0'}}, {'case_type': {'$eq': 'Crl.A'}}]}
  bm25_filter: {'year': '2025.0', 'case_type': 'Crl.A'}
  resolved PARSED_METADATA_FILTER: {'$and': [{'year': {'$eq': '2025.0'}}, {'case_type': {'$eq': 'Crl.A'}}]}
  ⚙️ Executing: dense_search({'query': 'Crl.P.L.A.645-L_2025 criminal law appeal', 'metadata_filter': {'$and': [{'year': {'$eq': '2025.0'}}, {'case_type': {'$eq': 'Crl.A'}}]}, 'k': 30})
  ⚙️ Executing: bm25_search({'q

In [52]:
result = run_agentic_system("What is Crl.P.L.A.645-L_2025 about?", verbose=True)


🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'What is Crl.P.L.A.645-L_2025 about?'})
  ⚙️ Executing: expand_case({'case_id': 'Crl.P.L.A.645_2025', 'k': 15})

⚠️ Plan 1 failed: expand_case: no chunks found for Crl.P.L.A.645_2025

📋 Generating Plan 2...
  📋 Replan generated: ['expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: expand_case({'case_id': 'SC-PK_Crl.P.L.A.645_2025', 'k': 15})

FINAL RESULT:
Context insufficient. Answer could not be verified after replanning.


this broke

In [53]:
result = run_agentic_system("cases about cryptocurrency regulation in Pakistan", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'cases about cryptocurrency regulation in Pakistan'})
  🔍 Parsed intent: {'semantic_query': 'cryptocurrency regulation pakistan crypto bitcoin blockchain fintech law', 'metadata_filter': None}
  normalized_filter: None
  chroma_filter: None
  bm25_filter: None
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'cryptocurrency regulation pakistan crypto bitcoin blockchain fintech law', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'cryptocurrency regulation pakistan crypto bitcoin blockchain fintech law', 'bm25_filter': None, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: C.P.L.A.278_2023
   Score: 0.0328
   Methods: DENSE, BM

In [54]:
result = run_agentic_system("Why did Justice Yahya Afridi rule against property rights in C.A.875_2017?", verbose=True)


🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'Why did Justice Yahya Afridi rule against property rights in C.A.875_2017?'})
  ⚙️ Executing: expand_case({'case_id': 'C.A.875_2017', 'k': 15})
  🔧 Resolved case_id: C.A.875_2017 → SC-PK_C.A.875_2017
  ⚙️ Executing: generate_answer({})
  ⚙️ Executing: verify_answer({})
  ✅ VERDICT: YES

FINAL RESULT:
Selected Case: SC-PK_C.A.875_2017

Answer: Justice Yahya Afridi ruled against the property rights of the contesting respondents in C.A.875_2017 because their application under the Redemption and Restitution of Mortgaged Lands Act, 1964, was time-barred, and the Board of Revenue had no power to extend the period of limitation under the Act retrospectively, thereby reviving a claim that had become time-barred.

Case Id: SC-PK_C.A.875_2017

Relevant chunks: "We invited learned counsel for

In [57]:
result = run_agentic_system("What is Crl.P.L.A.645-L_2025 about?", verbose=True)


🔀 Router: qa

💬 QA AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['detect_case_reference', 'expand_case', 'generate_answer', 'verify_answer']
  ⚙️ Executing: detect_case_reference({'query': 'What is Crl.P.L.A.645-L_2025 about?'})


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


  ⚙️ Executing: expand_case({'case_id': 'Crl.P.L.A.645-L_2025', 'k': 15})
✅ DB loaded: 9226 chunks
  ⚙️ Executing: generate_answer({})
  ⚙️ Executing: verify_answer({})
  ✅ VERDICT: YES

FINAL RESULT:
Selected Case: Crl.P.L.A.645-L_2025

Answer: Crl.P.L.A.645-L_2025 is about the enforcement of judicial orders regarding pre-arrest bail and the prompt arrest of accused persons after the dismissal of their pre-arrest bail applications, emphasizing that investigating authorities must act upon court orders without delay.

Case Id: Crl.P.L.A.645-L_2025

Relevant chunks: "It must be remembered that interim protection is not automatic; it must be specifically sought and expressly granted. Absent such an order, a refusal of bail remains fully operative and must be implemented promptly and in good faith by investigating authorities.", "This Court, therefore, finds it imperative to state clearly that investigating officers and police authorities are legally bound to act upon court orders dismissi

In [58]:
result = run_agentic_system("I need cases where the judge is Ayesha Malik", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'I need cases where the judge is Ayesha Malik'})
  🔍 Parsed intent: {'semantic_query': 'Justice Ayesha Malik', 'metadata_filter': {'judges': 'MR. JUSTICE AYESHA MALIK'}}
  chroma items: []
  normalized_filter: {'judges': 'AYESHA MALIK'}
  chroma_filter: None
  bm25_filter: {'judges': 'AYESHA MALIK'}
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'Justice Ayesha Malik', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'Justice Ayesha Malik', 'bm25_filter': {'judges': 'AYESHA MALIK'}, 'k': 30})
  ⚠️ BM25: No chunks matched metadata filter
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: C.P.L.A.240_2021
   Score: 0.0164
   Methods: DENSE


In [59]:
result = run_agentic_system("I need cases where the judge is Mrs. Justice Ayesha", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'I need cases where the judge is Mrs. Justice Ayesha'})
  🔍 Parsed intent: {'semantic_query': 'Mrs. Justice Ayesha', 'metadata_filter': {'judges': 'MRS. JUSTICE AYESHA'}}
  chroma items: []
  normalized_filter: {'judges': 'AYESHA'}
  chroma_filter: None
  bm25_filter: {'judges': 'AYESHA'}
  resolved PARSED_METADATA_FILTER: None
  ⚙️ Executing: dense_search({'query': 'Mrs. Justice Ayesha', 'metadata_filter': None, 'k': 30})
  ⚙️ Executing: bm25_search({'query': 'Mrs. Justice Ayesha', 'bm25_filter': {'judges': 'AYESHA'}, 'k': 30})
  ⚙️ Executing: aggregate_rrf({})
  ⚙️ Executing: filter_relative({})

FINAL RESULT:
Found Cases:

1. Case ID: C.P.L.A.240_2021
   Score: 0.0323
   Methods: DENSE, BM25
   Preview: SUPREME COURT OF PAKISTAN
(Appellate Jurisdict

In [63]:
result = run_agentic_system("I need Criminal appeals by Justice Munib Akhtar from 2024", verbose=True)


🔀 Router: case_search

🔍 CASE SEARCH AGENT executing...

📋 Generating Plan 1...
  📋 Plan generated: ['parse_legal_query', 'dense_search', 'bm25_search', 'aggregate_rrf', 'filter_relative']
  ⚙️ Executing: parse_legal_query({'user_query': 'I need Criminal appeals by Justice Munib Akhtar from 2024'})
  🔍 Parsed intent: {'semantic_query': 'Criminal appeals Justice Munib Akhtar', 'metadata_filter': {'judges': 'MR. JUSTICE MUNIB AKHTAR', 'case_type': 'Crl.A', 'year': '2024.0'}}
  chroma items: [('case_type', 'Crl.A'), ('year', '2024.0')]
  normalized_filter: {'judges': 'MUNIB AKHTAR', 'case_type': 'Crl.A', 'year': '2024.0'}
  chroma_filter: {'$and': [{'case_type': {'$eq': 'Crl.A'}}, {'year': {'$eq': '2024.0'}}]}
  bm25_filter: {'judges': 'MUNIB AKHTAR', 'case_type': 'Crl.A', 'year': '2024.0'}
  resolved PARSED_METADATA_FILTER: {'$and': [{'case_type': {'$eq': 'Crl.A'}}, {'year': {'$eq': '2024.0'}}]}
  ⚙️ Executing: dense_search({'query': 'Criminal appeals Justice Munib Akhtar', 'metadata_fil

## what if question is from multiple documents? what if one document has answer but question mentions multiple? do you optimise for this or just say this is specific RAG not general legal QA RAG